In [2]:
from pathlib import Path
import pandas as pd
%run "01_EX_Clase_Jugador.ipynb"

# 1. CARGA DE DATOS
Ruta_datos = Path(r"C:\proyecto-faust\proyecto-nba-analytics\data\complete_nba_final_dataset_2026.csv")
df_nba = pd.read_csv(Ruta_datos)

# fillna(0): Rellenamos huecos vacíos con ceros para evitar errores de cálculo
df_nba.fillna(0, inplace=True)

lista_jugadores = []

# 2. BUCLE DE INSTANCIACIÓN E INYECCIÓN
for index, fila in df_nba.iterrows():
    try:
        # --- A. Datos Identificativos (Constructor) ---
        nombre = fila['DISPLAY_FIRST_LAST']
        nacionalidad = fila['COUNTRY']
        altura = fila['HEIGHT']
        posicion = fila['POSITION']
        equipo = fila['TEAM_ABBREVIATION']
        edad = fila['Edad']

        # Instanciamos el objeto
        jugador_obj = Jugador(nombre, edad, nacionalidad, altura, posicion, equipo)

        # --- B. Mapeo de Estadísticas Directas ---
        jugador_obj.puntos = fila['PTS']
        jugador_obj.asistencias = fila['AST']
        jugador_obj.rebotes = fila['REB']
        jugador_obj.robos = fila['STL']
        jugador_obj.tapones = fila['BLK']
        jugador_obj.perdidas = fila['TOV']
        
        # Nuevas columnas críticas que añadimos en el último paso
        jugador_obj.partidos_jugados = int(fila['GP'])
        jugador_obj.faltas = fila['PF']
        jugador_obj.partidos_lesionado = int(fila['Partidos_Lesionado'])
        jugador_obj.minutos = fila['MIN'] # Puede venir como número o string, lo trataremos luego

        # --- C. Cálculo de Métricas Derivadas (Tiros Fallados) ---
        # Tu clase requiere los fallos, el CSV trae Intentos (A) y Aciertos (M)
        
        # Triples
        jugador_obj.triples_anotados = fila['FG3M']
        jugador_obj.triples_fallados = fila['FG3A'] - fila['FG3M']
        
        # Tiros Libres
        jugador_obj.tiros_libres_anotados = fila['FTM']
        jugador_obj.tiros_libres_fallados = fila['FTA'] - fila['FTM']
        
        # Tiros de 2 (Campo total - Triples)
        tiros_dos_intentados = fila['FGA'] - fila['FG3A']
        jugador_obj.tiros_dos_anotados = fila['FGM'] - fila['FG3M']
        jugador_obj.tiros_dos_fallados = tiros_dos_intentados - jugador_obj.tiros_dos_anotados

        # --- D. Ejecución de Lógica de Negocio ---
        # Calculamos la puntuación usando tu método personalizado
        # Nota: Guardamos el resultado en un atributo dinámico para poder ordenar después
        jugador_obj.valoracion_final = jugador_obj.calcular_eficiencia_jugador()

        lista_jugadores.append(jugador_obj)

    except Exception as e:
        print(f"[ERROR] Fallo al importar {fila.get('DISPLAY_FIRST_LAST', 'Desconocido')}: {e}")
        
# 3. PERSISTENCIA 
# Convertimos la lista de OBJETOS a un DATAFRAME de Pandas.
# Usamos 'vars(x)' que extrae los atributos de cada objeto como un diccionario.
df_procesado = pd.DataFrame([vars(j) for j in lista_jugadores])

# Guardamos en formato PICKLE (mejor que CSV porque mantiene los tipos de datos exactos)
ruta_guardado = Path(r"C:\proyecto-faust\proyecto-nba-analytics\data\nba_processed_dataset.pkl")
df_procesado.to_pickle(ruta_guardado)